In [ ]:
import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature

import pandas as pd
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

In [ ]:
# -----------------------------
# 1. Load & prepare data
# -----------------------------
data = load_diabetes()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name="target")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
# -----------------------------
# 2. Setup MLflow experiment
# -----------------------------
mlflow.set_experiment("diabetes_rf_experiment")

In [ ]:
# -----------------------------
# 3. Train & log with MLflow
# -----------------------------
with mlflow.start_run(run_name="random_forest_run") as run:
    
    # Model
    model = RandomForestRegressor(
        n_estimators=100, max_depth=6, random_state=42
    )
    model.fit(X_train, y_train)

    # Predictions
    y_pred = model.predict(X_test)

    # Metrics
    mse = mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    # Log params, metrics
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("max_depth", 6)
    mlflow.log_metric("mse", mse)
    mlflow.log_metric("r2", r2)

    # Log model
    signature = infer_signature(X_train, model.predict(X_train))
    mlflow.sklearn.log_model(
        sk_model=model,
        name="model",
        signature=signature,
        input_example=X_train.head(1),
        registered_model_name="DiabetesRandomForestModel"
    )

    print(f"Run ID: {run.info.run_id}")
    print(f"Model logged and registered in MLflow Model Registry!")

In [ ]:
# -----------------------------
# 4. Loading the model back
# -----------------------------
# You can load a specific version from the registry like this:
model_name = "DiabetesRandomForestModel"
model_version = 1  # first registered version
loaded_model = mlflow.pyfunc.load_model(
    model_uri=f"models:/{model_name}/{model_version}"
)

# Test loaded model
sample = X_test.head(5)
print("Predictions from loaded model:", loaded_model.predict(sample))
